# Minimal Statistical Test

Only the essentials:
1. Set `DATASETS` (two files)
2. Set `METRIC_KEY`
3. Run the notebook


In [13]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from TSB_AD.evaluation.metrics import get_metrics
from TSB_AD.model_wrapper import run_Unsupervise_AD

warnings.filterwarnings("ignore")

## Configuration (Edit This Cell)

In [14]:
METHOD_NAME = "Sub_PCA"

DATASETS = [
    Path("../datasets/TSB-AD-U/001_NAB_id_1_Facility_tr_1007_1st_2014.csv"),
    Path("../datasets/TSB-AD-U/002_NAB_id_2_WebService_tr_1500_1st_4106.csv"),
]

METRIC_KEY = "AUC-ROC"  # Example alternatives: "AUC-PR", "PA-F1", "VUS-PR"

print(f"Method: {METHOD_NAME}")
print(f"Datasets: {[p.name for p in DATASETS]}")
print(f"Metric: {METRIC_KEY}")

Method: Sub_PCA
Datasets: ['001_NAB_id_1_Facility_tr_1007_1st_2014.csv', '002_NAB_id_2_WebService_tr_1500_1st_4106.csv']
Metric: AUC-ROC


In [15]:
rows = []

for dataset_path in DATASETS:
    df = pd.read_csv(dataset_path).dropna()
    data = df.iloc[:, :-1].values.astype(float)
    labels = df["Label"].astype(int).to_numpy()

    scores = run_Unsupervise_AD(METHOD_NAME, data, periodicity=1)
    if isinstance(scores, str):
        raise RuntimeError(scores)

    scores = np.asarray(scores).ravel()
    n = min(len(scores), len(labels))
    scores = scores[:n]
    labels = labels[:n]

    metrics = get_metrics(scores, labels)
    if METRIC_KEY not in metrics:
        raise ValueError(f"Metric '{METRIC_KEY}' not found. Available: {list(metrics.keys())}")

    rows.append(
        {
            "dataset": dataset_path.name,
            "method": METHOD_NAME,
            METRIC_KEY: float(metrics[METRIC_KEY]),
        }
    )

results_df = pd.DataFrame(rows)
display(results_df)

,dataset,method,AUC-ROC
0,001_NAB_id_1_Facility_tr_1007_1st_2014.csv,Sub_PCA,0.505614
1,002_NAB_id_2_WebService_tr_1500_1st_4106.csv,Sub_PCA,0.710565


In [16]:
print(f"Mean {METRIC_KEY}: {results_df[METRIC_KEY].mean():.4f}")

Mean AUC-ROC: 0.6081
